In [2]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

from transformers import pipeline

# Now it will load much faster because it skips TensorFlow entirely
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

config.json: 0.00B [00:00, ?B/s]

d:\Anaconda\envs\tweet_project\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nabil\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [3]:
import pandas as pd
import os

# Your exact local file path
file_path = r"F:\research\chatbot\D\Multi-modal-codes\data\CrisisMMD\crisismmd_datasplit_all\crisismmd_datasplit_agreed_label\task_humanitarian_text_img_agreed_lab_dev.tsv"

# 1. Check if the file exists
if not os.path.exists(file_path):
    print(f"Error: Could not find the file at {file_path}")
    print("Please check if the drive is connected or the path is spelled correctly.")
else:
    print("Dataset found! Loading data...\n")
    
    # 2. Load the TSV (Tab-Separated Values) dataset
    # We use sep='\t' because it's a .tsv file, not a comma-separated .csv
    df = pd.read_csv(file_path, sep='\t')
    
    # 3. Print the columns to understand the structure
    print("--- Dataset Columns ---")
    print(df.columns.tolist())
    print("\n--- First 3 Rows ---")
    print(df.head(3))
    
    # 4. Analyze the Humanitarian Categories (The core of your paper)
    # The label column in CrisisMMD is usually named 'label_text_image' or similar. 
    # We will try to find the label column dynamically.
    label_col = [col for col in df.columns if 'label' in col.lower()]
    
    if label_col:
        target_col = label_col[0]
        print(f"\n--- Class Distribution for '{target_col}' ---")
        distribution = df[target_col].value_counts()
        print(distribution)
        
        # Calculate percentages to see exactly how imbalanced 'Affected Individuals' is
        print("\n--- Class Percentages ---")
        print((distribution / len(df)) * 100)
    else:
        print("\nCould not automatically identify the label column. Please check the column names above.")

Dataset found! Loading data...

--- Dataset Columns ---
['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image', 'label', 'label_text', 'label_image', 'label_text_image']

--- First 3 Rows ---
             event_name            tweet_id              image_id  \
0  california_wildfires  920329866901381123  920329866901381123_0   
1        hurricane_irma  909744783115120640  909744783115120640_0   
2        hurricane_irma  909925247331241984  909925247331241984_0   

                                          tweet_text  \
0  Family Miraculously Finds Dog They Lost When E...   
1  Hurricane Irma on #Twitter https://t.co/HDssyS...   
2  i feel attacked by jin in so many levels how t...   

                                               image  \
0  data_image/california_wildfires/17_10_2017/920...   
1  data_image/hurricane_irma/18_9_2017/9097447831...   
2  data_image/hurricane_irma/18_9_2017/9099252473...   

                        label                  label_text  \
0            n

In [ ]:
!pip install -U diffusers
import torch
from diffusers import StableDiffusionPipeline
import os
import pandas as pd
from tqdm import tqdm

# 1. Setup the Generator (optimized for RTX 3050 4GB VRAM)
print("Loading Stable Diffusion Pipeline...")
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 # Crucial for 4GB VRAM
)
pipe = pipe.to("cuda")
# Optional: Enable memory efficient attention if you run into VRAM limits
pipe.enable_attention_slicing() 

# 2. Define the Prompts for "Affected Individuals"
# We need variety: floods, earthquakes, wildfires
prompts = [
    "A realistic photograph of displaced people carrying their belongings through a flooded street, heavy rain, disaster zone.",
    "A news photo of a family sitting outside a destroyed home after an earthquake, looking distressed, debris everywhere.",
    "A high quality picture of tired volunteers handing out blankets to affected individuals in a disaster relief camp.",
    "A photo of people evacuating a neighborhood during a severe wildfire, smoke in the background, carrying bags."
]

# 3. Setup output directories
output_dir = "synthetic_data/images"
os.makedirs(output_dir, exist_ok=True)

# 4. Generate the Data (Let's start with a small batch of 12 images to test)
# For the final paper, you will scale this up to ~500
num_images_per_prompt = 3 
synthetic_records = []
image_counter = 0

print(f"\nGenerating {len(prompts) * num_images_per_prompt} synthetic images...")

for prompt in prompts:
    for i in tqdm(range(num_images_per_prompt), desc=f"Generating for prompt: '{prompt[:30]}...'"):
        
        # Generate the image
        image = pipe(prompt, num_inference_steps=25).images[0]
        
        # Create a unique filename
        filename = f"synth_affected_{image_counter}.png"
        filepath = os.path.join(output_dir, filename)
        
        # Save the image
        image.save(filepath)
        
        # Record the metadata to merge with your TSV later
        synthetic_records.append({
            "event_name": "synthetic_disaster",
            "tweet_id": f"synth_{image_counter}",
            "image_id": f"synth_{image_counter}_0",
            "tweet_text": prompt, # Using the prompt as the synthetic "tweet"
            "image": filepath,
            "label": "affected_individuals",
            "label_text": "affected_individuals",
            "label_image": "affected_individuals",
            "label_text_image": "Positive"
        })
        
        image_counter += 1

# 5. Save the synthetic dataset log
df_synthetic = pd.DataFrame(synthetic_records)
csv_path = "synthetic_data/synthetic_affected_individuals.tsv"
df_synthetic.to_csv(csv_path, sep='\t', index=False)

print(f"\nDone! Images saved to '{output_dir}'.")
print(f"Metadata saved to '{csv_path}'.")

   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   -------------------------------

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

d:\Anaconda\envs\tweet_project\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nabil\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 061858dc-4925-42ab-b2db-440880939803)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/vae%2Fconfig.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: d734c957-b33a-4acb-8ac8-fbf92cf9be93)')' thrown while requesting GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/unet%2Fconfig.json
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ef97b2d3-2971-4c38-a6ba-bddc91bcd8f8)')' thrown while requesting HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]